# 07-units-fire2 -- Dimensionally constrained PM2.5 search on the corrected fire data (Bogota)

On 2026-10-20 the gridded fire product was found to be spatially wrong in most years and was replaced by a FIRMS regeneration
(`Scripts/colombia/rebuild_fire_columns.py`). The PM2.5 feature selection (`06_feature_selection.ipynb`) was re-run on the corrected
master dataset and now selects: w700, CO, fire NE (lag 2), sensible heat flux (lag 1), wind speed, fire N, sensible heat flux, boundary-layer
height, fire E (lag 2). This notebook runs the constrained PySR search on that pool: same operators, `niterations=200`, `populations=24`,
`maxsize=25`, three seeds, units as elsewhere (gases `kg/m^3`; sensible heat flux `W/m^2`; BLH `m`; fire counts dimensionless).
The rows labelled `free-form` in the scored table are the *old* Eq. 1 front (old pool) re-scored by the same leave-one-year-out
procedure on the corrected data, as a reference.

In [1]:
# %% Settings (edit here, then Run All)
CITY, TARGET, TAG = "bogota", "pm25", "units_fire2"          # fire2 = corrected FIRMS fire product (2026-10-20)
MASTER = "bogota_master_dataset.parquet"
SELECTION_JSON = "bogota_feature_selection.json"            # selected_no_lag = the pool (re-selected with the corrected fire data)
EXTRA_FEATURES = []
DROP_FEATURES = []
SEEDS = [0, 1, 2]
PYSR_KW = dict(niterations=200, binary_operators=["+", "-", "*", "/"], unary_operators=["log", "sqrt", "exp"],
               model_selection="best", maxsize=25, populations=24)     # identical to every constrained run in the paper
UNIT_MAP = {"w700": "Pa/s", "co": "kg/m^3", "wind_speed_era5": "m/s", "blh_mean": "m",
            "sensible_heat_flux": "W/m^2", "sensible_heat_flux_lag1": "W/m^2",      # ERA5 sshf / 3600
            "hotspots_sector_NE_lag2": "", "hotspots_sector_N": "", "hotspots_sector_E_lag2": ""}   # fire counts: dimensionless
Y_UNITS = "kg/m^3"
FREEFORM_FRONT = "bogota_pm25_pysr_units_pareto_front.csv"   # rows labelled 'free-form' here = the OLD Eq.1 front, re-scored on the corrected fire data (reference)
MAX_COMPLEXITY_REPORTED = 25

In [2]:
import sys, json, re, warnings
from pathlib import Path
PROJECT_ROOT = Path("/path/to/project")
sys.path.insert(0, str(PROJECT_ROOT / "Scripts"))
import numpy as np, pandas as pd, sympy as sp
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score
DATA_DIR = PROJECT_ROOT / "Data" / "processed" / "colombia" / CITY
df = pd.read_parquet(DATA_DIR / MASTER)
df["date"] = pd.to_datetime(df["date"]); df["year"] = df["date"].dt.year
df = df.dropna(subset=[TARGET]).reset_index(drop=True)
selection = json.load(open(DATA_DIR / SELECTION_JSON))
features = [f for f in selection["selected_no_lag"] if f not in DROP_FEATURES]
for f in EXTRA_FEATURES:
    if f not in features: features.append(f)
print("pool:", features)
data = df.dropna(subset=features + [TARGET]).reset_index(drop=True)
X = data[features].to_numpy(); y = data[TARGET].to_numpy()
X_units = [UNIT_MAP[f] for f in features]
print(data.shape, dict(zip(features, X_units)), "years:", sorted(data.year.unique()))

pool: ['w700', 'co', 'hotspots_sector_NE_lag2', 'sensible_heat_flux_lag1', 'wind_speed_era5', 'hotspots_sector_N', 'sensible_heat_flux', 'blh_mean', 'hotspots_sector_E_lag2']
(3283, 216) {'w700': 'Pa/s', 'co': 'kg/m^3', 'hotspots_sector_NE_lag2': '', 'sensible_heat_flux_lag1': 'W/m^2', 'wind_speed_era5': 'm/s', 'hotspots_sector_N': '', 'sensible_heat_flux': 'W/m^2', 'blh_mean': 'm', 'hotspots_sector_E_lag2': ''} years: [np.int32(2014), np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022)]


In [3]:
from pysr import PySRRegressor
fronts = {}
for seed in SEEDS:
    sr = PySRRegressor(**PYSR_KW, random_state=seed)
    sr.fit(X, y, variable_names=features, X_units=X_units, y_units=Y_UNITS)
    f = sr.equations_[["complexity", "loss", "equation"]].copy(); f["seed"] = seed
    f.to_csv(DATA_DIR / f"{CITY}_{TARGET}_pysr_{TAG}_pareto_front_seed{seed}.csv", index=False)
    fronts[seed] = f
    print("seed", seed, "done; best:", sr.get_best()["equation"])

Compiling Julia backend...


/path/to/project/.venv/lib/python3.11/site-packages/pysr/sr.py:2385: UserWarning: Note: Setting `random_state` without also setting `deterministic=True` and `parallelism='serial'` will result in non-deterministic searches.
  warnings.warn(


┌ Warning: You are using dimensional constraints, but `dimensional_constraint_penalty` was not set. The default penalty of `1000.0` will be used.
└ @ SymbolicRegression ~/.julia/packages/SymbolicRegression/O7o63/src/Configure.jl:73
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:


Expressions evaluated per second: 8.140e+03
Progress: 20 / 200 total iterations (10.229%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           6.443e+01  0.000e+00  y[m⁻³ kg] = 18.301[?]
3           3.988e+01  2.399e-01  y[m⁻³ kg] = co[m⁻³ kg] * 23.843[?]
5           3.984e+01  4.988e-04  y[m⁻³ kg] = (co[m⁻³ kg] * 22.869[?]) - -0.98945[?]
7           3.538e+01  5.927e-02  y[m⁻³ kg] = co[m⁻³ kg] * ((7.6412[?] / wind_speed_...
                                              era5[m s⁻¹]) + 15.118[?])
8           2.627e+01  2.977e-01  y[m⁻³ kg] = sqrt(hotspots_sector_E_lag2 * 0.12583[...
                                              ?]) + (co[m⁻³ kg] * 17.745[?])
10          2.604e+01  4.415e-03  y[m⁻³ kg] = (co[m⁻³ kg] * 17.533[?]) + sqrt(co[m⁻³...
                                           

/path/to/project/.venv/lib/python3.11/site-packages/pysr/sr.py:2385: UserWarning: Note: Setting `random_state` without also setting `deterministic=True` and `parallelism='serial'` will result in non-deterministic searches.
  warnings.warn(


┌ Warning: You are using dimensional constraints, but `dimensional_constraint_penalty` was not set. The default penalty of `1000.0` will be used.
└ @ SymbolicRegression ~/.julia/packages/SymbolicRegression/O7o63/src/Configure.jl:73
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:
/path/to/project/.venv/lib/python3.11/site-packages/pysr/sr.py:2385: UserWarning: Note: Setting `random_state` without also setting `deterministic=True` and `parallelism='serial'` will result in non-deterministic searches.
  warnings.warn(



Expressions evaluated per second: 9.740e+03
Progress: 22 / 200 total iterations (11.229%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           6.443e+01  0.000e+00  y[m⁻³ kg] = 18.291[?]
3           3.988e+01  2.399e-01  y[m⁻³ kg] = co[m⁻³ kg] * 23.844[?]
5           3.985e+01  3.803e-04  y[m⁻³ kg] = (co[m⁻³ kg] * 23.388[?]) + 0.5527[?]
7           2.657e+01  2.026e-01  y[m⁻³ kg] = (hotspots_sector_E_lag2 * 0.0082147[?]...
                                              ) - (co[m⁻³ kg] * -20.752[?])
10          2.318e+01  4.557e-02  y[m⁻³ kg] = (co[m⁻³ kg] * 17.36[?]) + (sqrt(hotspo...
                                              ts_sector_NE_lag2 / 17.73[?]) / wind_speed_era5[m ...
                                              s⁻¹])
12          2.310e+01  1.679e-03  y[m⁻³ kg] = (co[m⁻³


Expressions evaluated per second: 9.710e+03
Progress: 24 / 200 total iterations (12.333%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           6.443e+01  0.000e+00  y[m⁻³ kg] = 18.311[?]
3           3.988e+01  2.399e-01  y[m⁻³ kg] = co[m⁻³ kg] * 23.845[?]
5           3.983e+01  5.450e-04  y[m⁻³ kg] = (co[m⁻³ kg] * 22.33[?]) + 1.1028[?]
6           3.439e+01  1.470e-01  y[m⁻³ kg] = sqrt(co[m⁻³ kg] / wind_speed_era5[m s⁻...
                                              ¹]) * 21.553[?]
7           2.833e+01  1.940e-01  y[m⁻³ kg] = (co[m⁻³ kg] * 21.111[?]) + (hotspots_s...
                                              ector_E_lag2 * 0.0052755[?])
8           2.628e+01  7.489e-02  y[m⁻³ kg] = (co[m⁻³ kg] * 18.108[?]) + sqrt(hotspo...
                                              ts_sector_E

┌ Warning: You are using dimensional constraints, but `dimensional_constraint_penalty` was not set. The default penalty of `1000.0` will be used.
└ @ SymbolicRegression ~/.julia/packages/SymbolicRegression/O7o63/src/Configure.jl:73
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:


In [4]:
FLOAT = re.compile(r"(?<![A-Za-z_0-9.])\d+\.\d+(?:[eE][+-]?\d+)?")
def make_model(eq, feats):
    vals = []
    def sub(m): vals.append(float(m.group(0))); return f"c{len(vals)-1}"
    s = FLOAT.sub(sub, eq)
    syms = {f: sp.Symbol(f) for f in feats}; cs = [sp.Symbol(f"c{i}") for i in range(len(vals))]
    expr = sp.sympify(s, locals={**syms, **{str(c): c for c in cs}, "sqrt": sp.sqrt, "log": sp.log, "exp": sp.exp})
    used = [f for f in feats if syms[f] in expr.free_symbols]
    keep = [i for i, c in enumerate(cs) if c in expr.free_symbols]
    fn = sp.lambdify([syms[f] for f in used] + [cs[i] for i in keep], expr, "numpy")
    p0 = np.array([vals[i] for i in keep])
    return (lambda Xt, *p: fn(*Xt, *p)), used, p0

yy = data[TARGET].to_numpy(float); years = data["year"].to_numpy(); YRS = sorted(set(years))
def score(eq, feats):
    model, used, p0 = make_model(eq, feats)
    if not used:                                   # constant equation: leave-one-year-out mean
        oof = np.concatenate([np.full((years == yr).sum(), np.mean(yy[years != yr])) for yr in YRS]); yo = np.concatenate([yy[years == yr] for yr in YRS])
        per = [r2_score(yy[years == yr], np.full((years == yr).sum(), np.mean(yy[years != yr]))) for yr in YRS]
        return dict(r2_insample=0.0, r2_pooled=r2_score(yo, oof), r2_mean_years=float(np.mean(per)), vars="")
    Xt = tuple(data[f].to_numpy(float) for f in used)
    with np.errstate(all="ignore"), warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try: popt, _ = curve_fit(model, Xt, yy, p0=p0, maxfev=20000)
        except Exception: popt = p0
        pred = np.full(len(yy), np.nan)
        for yr in YRS:
            te = years == yr
            try: pf, _ = curve_fit(model, tuple(a[~te] for a in Xt), yy[~te], p0=popt, maxfev=20000)
            except Exception: pf = popt
            pred[te] = model(tuple(a[te] for a in Xt), *pf)
        ok = np.all(np.isfinite(pred)); ins = model(Xt, *popt)
        per = [r2_score(yy[years == yr], pred[years == yr]) for yr in YRS] if ok else [np.nan]
        return dict(r2_insample=r2_score(yy, ins) if np.all(np.isfinite(ins)) else np.nan, r2_pooled=r2_score(yy, pred) if ok else np.nan,
                    r2_mean_years=float(np.mean(per)), vars=",".join(used))

rows = []
for seed, f in fronts.items():
    for _, r in f.iterrows():
        if r.complexity <= MAX_COMPLEXITY_REPORTED: rows.append(dict(seed=str(seed), complexity=int(r.complexity), equation=r.equation, **score(r.equation, features)))
ff = pd.read_csv(DATA_DIR / FREEFORM_FRONT).sort_values("complexity")
ff_feats = sorted({t for e in ff.equation for t in re.findall(r"[A-Za-z_][A-Za-z0-9_]*", e) if t in data.columns})
for _, r in ff.iterrows():
    if r.complexity <= MAX_COMPLEXITY_REPORTED:
        try: rows.append(dict(seed="free-form", complexity=int(r.complexity), equation=r.equation, **score(r.equation, ff_feats)))
        except Exception as e: print("free-form eq skipped", r.complexity, e)
scored = pd.DataFrame(rows); scored.to_csv(DATA_DIR / f"{CITY}_{TARGET}_pysr_{TAG}_scored.csv", index=False)
pd.set_option("display.width", 250, "display.max_colwidth", 110)
print("n days", len(data)); print(scored[["seed", "complexity", "r2_pooled", "r2_mean_years", "vars"]].round(3).to_string(index=False))

n days 3283
     seed  complexity  r2_pooled  r2_mean_years                                                                           vars
        0           1     -0.017         -0.084                                                                               
        0           3      0.370          0.341                                                                             co
        0           5      0.368          0.338                                                                             co
        0           6      0.465          0.439                                                             co,wind_speed_era5
        0           7      0.574          0.543                                                      co,hotspots_sector_E_lag2
        0           8      0.580          0.551                                                      co,hotspots_sector_E_lag2
        0           9      0.598          0.567                                      co,wind_speed_